# Sales Performance Dashboard | Data Analyst Project

**Project 1 | Retail and E-commerce**  
Tools: Python (pandas), Oracle SQL, Power BI

## Objective
Analyze retail order-line data to understand sales, profit, product performance, regional performance and monthly trends. Prepare clean data for Oracle SQL and Power BI.

**Dataset:** Sample Superstore retail dataset (classic educational dataset distributed through Kaggle and Tableau sample-data sources). It contains order dates, customers, geography, products, sales, quantity, discount and profit.

**Important:** Run every cell from top to bottom in Google Colab. The notebook downloads a public CSV mirror. If the link is unavailable, upload the CSV downloaded from the Kaggle dataset page and use the fallback cell.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

OUTPUT = Path('/content/project_outputs')
OUTPUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Load the dataset

The default link is a public mirror of the Sample Superstore CSV. It is the classic Kaggle/Tableau training dataset, not confidential company data.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/mikemooreviz/superstore/master/superstore.csv'
try:
    raw = pd.read_csv(DATA_URL, encoding='latin1')
    print('Dataset downloaded successfully.')
except Exception as e:
    print('Automatic download failed:', e)
    print('Use the upload cell below to upload Sample - Superstore.csv from Kaggle.')
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    raw = pd.read_csv(filename, encoding='latin1')

print('Rows and columns:', raw.shape)
display(raw.head())

## 3. Inspect and clean the data

Cleaning steps: standardise column names, remove exact duplicate rows, parse dates, convert numeric columns, remove rows missing essential analysis fields, and flag invalid values. Negative profit is retained because it represents a real business loss, not a data-entry error.

In [ ]:
df = raw.copy()
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]

# Standardise key column names (source file may use slightly different labels)
rename_map = {'Row_ID':'Row_ID', 'Order_ID':'Order_ID', 'Order_Date':'Order_Date', 'Ship_Date':'Ship_Date',
              'Customer_ID':'Customer_ID', 'Customer_Name':'Customer_Name', 'Product_ID':'Product_ID',
              'Product_Name':'Product_Name', 'Sub_Category':'Sub_Category', 'Postal_Code':'Postal_Code', 'State':'State_Name'}
df = df.rename(columns=rename_map)

rows_before = len(df)
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

for col in ['Order_Date', 'Ship_Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ['Sales', 'Quantity', 'Discount', 'Profit']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

required = ['Order_ID','Order_Date','Customer_ID','Product_ID','Region','Category','Sales','Quantity','Discount','Profit']
missing_required_before = int(df[required].isna().any(axis=1).sum())
df = df.dropna(subset=required).copy()
invalid_sales = int((df['Sales'] < 0).sum())
invalid_quantity = int((df['Quantity'] <= 0).sum())
invalid_discount = int(((df['Discount'] < 0) | (df['Discount'] > 1)).sum())
# Remove impossible sales/quantity/discount records; retain negative profit values.
df = df[(df['Sales'] >= 0) & (df['Quantity'] > 0) & (df['Discount'].between(0,1))].copy()

df['Order_Month'] = df['Order_Date'].dt.to_period('M').dt.to_timestamp()
df['Order_Year'] = df['Order_Date'].dt.year
df['Profit_Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], np.nan)
df['Ship_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days if 'Ship_Date' in df.columns else np.nan

quality = pd.DataFrame({
    'Check':['Rows in raw file','Exact duplicates removed','Rows with missing required fields removed','Negative sales rows detected','Non-positive quantity rows detected','Discount outside 0-1 detected','Rows in cleaned file'],
    'Count':[rows_before,duplicates_removed,missing_required_before,invalid_sales,invalid_quantity,invalid_discount,len(df)]
})
display(quality)
print('Cleaned shape:', df.shape)
display(df.head())

## 4. KPI calculations

Definitions:
- **Total Sales:** sum of Sales across order lines.
- **Total Profit:** sum of Profit.
- **Total Orders:** distinct Order_ID count (not line-item count).
- **Customers:** distinct Customer_ID count.
- **Average Order Value:** Total Sales divided by distinct orders.
- **Profit Margin:** Total Profit divided by Total Sales.

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order_ID'].nunique()
total_customers = df['Customer_ID'].nunique()
quantity_sold = df['Quantity'].sum()
avg_order_value = total_sales / total_orders if total_orders else 0
profit_margin = total_profit / total_sales if total_sales else 0

kpis = pd.DataFrame({'KPI':['Total Sales','Total Profit','Total Orders','Total Customers','Quantity Sold','Average Order Value','Profit Margin'],
                     'Value':[total_sales,total_profit,total_orders,total_customers,quantity_sold,avg_order_value,profit_margin]})
display(kpis)

## 5. Sales analysis by month, region, category and product

In [ ]:
monthly_sales = df.groupby('Order_Month', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order_ID','nunique')).sort_values('Order_Month')
region_sales = df.groupby('Region', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order_ID','nunique')).sort_values('Sales', ascending=False)
category_sales = df.groupby('Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Quantity=('Quantity','sum')).sort_values('Sales', ascending=False)
product_sales = df.groupby(['Product_ID','Product_Name'], as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Quantity=('Quantity','sum')).sort_values('Sales', ascending=False)
top10_products = product_sales.head(10)

display(monthly_sales.head())
display(region_sales)
display(category_sales)
display(top10_products)

## 6. Visualisations

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(monthly_sales['Order_Month'], monthly_sales['Sales'], marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month'); plt.ylabel('Sales'); plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
plt.bar(region_sales['Region'], region_sales['Sales'])
plt.title('Sales by Region'); plt.xlabel('Region'); plt.ylabel('Sales'); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
plt.bar(category_sales['Category'], category_sales['Sales'])
plt.title('Sales by Category'); plt.xlabel('Category'); plt.ylabel('Sales'); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,6))
plt.barh(top10_products['Product_Name'].iloc[::-1], top10_products['Sales'].iloc[::-1])
plt.title('Top 10 Products by Sales'); plt.xlabel('Sales'); plt.tight_layout(); plt.show()

## 7. Business insights

The next cell calculates the observations from the dataset instead of hard-coding results. Use the printed values in your final report.

In [ ]:
best_region = region_sales.iloc[0]
best_category = category_sales.iloc[0]
worst_profit_category = category_sales.sort_values('Profit').iloc[0]
worst_subcats = df.groupby('Sub_Category', as_index=False)['Profit'].sum().sort_values('Profit').head(5) if 'Sub_Category' in df.columns else pd.DataFrame()
loss_lines = int((df['Profit'] < 0).sum())
loss_share = loss_lines / len(df) if len(df) else 0

print(f'Total sales: ${total_sales:,.2f}')
print(f'Total profit: ${total_profit:,.2f}')
print(f'Profit margin: {profit_margin:.2%}')
print(f'Highest-sales region: {best_region.Region} (${best_region.Sales:,.2f})')
print(f'Highest-sales category: {best_category.Category} (${best_category.Sales:,.2f})')
print(f'Lowest-profit category: {worst_profit_category.Category} (${worst_profit_category.Profit:,.2f} profit)')
print(f'Loss-making order lines: {loss_lines:,} ({loss_share:.2%} of cleaned lines)')
print('Five lowest-profit sub-categories:')
display(worst_subcats)

## 8. Validation tests

The project checks that totals reconcile between the cleaned data and grouped summaries. These tests should pass before you submit.

In [ ]:
assert not df.duplicated().any(), 'Duplicate rows remain.'
assert df[required].notna().all().all(), 'Required fields contain nulls.'
assert (df['Sales'] >= 0).all(), 'Negative sales remain.'
assert (df['Quantity'] > 0).all(), 'Non-positive quantity remains.'
assert df['Discount'].between(0,1).all(), 'Invalid discount remains.'
assert abs(monthly_sales['Sales'].sum() - total_sales) < 0.01, 'Monthly sales do not reconcile.'
assert abs(region_sales['Sales'].sum() - total_sales) < 0.01, 'Regional sales do not reconcile.'
assert abs(category_sales['Sales'].sum() - total_sales) < 0.01, 'Category sales do not reconcile.'
print('PASS: all data-quality and sales-reconciliation tests passed.')

## 9. Export files for Oracle SQL and Power BI

The cleaned CSV is the main Power BI import file. The summary files can also be imported for quick cross-checking.

In [ ]:
export_cols = [c for c in df.columns if c not in []]
df[export_cols].to_csv(OUTPUT/'superstore_cleaned.csv', index=False)
monthly_sales.to_csv(OUTPUT/'monthly_sales.csv', index=False)
region_sales.to_csv(OUTPUT/'region_sales.csv', index=False)
category_sales.to_csv(OUTPUT/'category_sales.csv', index=False)
top10_products.to_csv(OUTPUT/'top10_products.csv', index=False)
quality.to_csv(OUTPUT/'data_quality_report.csv', index=False)
kpis.to_csv(OUTPUT/'kpi_summary.csv', index=False)
print('Files created:')
for p in sorted(OUTPUT.glob('*.csv')): print(p.name, p.stat().st_size, 'bytes')

## 10. Download project outputs

Download the cleaned dataset and reports from Colab. These are the files to import into Power BI and use for Oracle SQL staging.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/sales_dashboard_outputs', 'zip', OUTPUT)
files.download('/content/sales_dashboard_outputs.zip')

## Conclusion

This project uses pandas to clean and analyse retail sales data, validates sales totals across monthly, regional and category summaries, and exports Power BI/Oracle-ready files. The final business conclusions must be based on the output values generated when the notebook is run.